---
title: Explore ATL06 with async-hdf5
---

This notebook opens an ICESat-2 ATL06 (land ice elevation) granule directly
from NASA's cloud archives over HTTPS using
[async-hdf5](https://github.com/maxrjones/async-hdf5). No data is
downloaded — HDF5 metadata and array chunks are fetched on demand via
byte-range requests.

## Find an ATL06 granule

We use [earthaccess](https://github.com/nsidc/earthaccess) to search NASA's
Common Metadata Repository (CMR) and obtain an authentication token.

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="Numcodecs codecs are not in the Zarr")
warnings.filterwarnings(
    "ignore", category=UserWarning, message=".*does not have a Zarr V3 specification.*"
)
warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="tqdm")

In [ ]:
import earthaccess

auth = earthaccess.login()

results = earthaccess.search_data(
    short_name="ATL06",
    cloud_hosted=True,
    bounding_box=(-55, 60, -30, 85),  # Greenland
    temporal=("2024-01-01", "2024-01-15"),
    count=1,
)

granule = results[0]
print(f"Granule: {granule['umm']['GranuleUR']}")

Get the HTTPS URL and set up an authenticated HTTP store.

In [ ]:
from urllib.parse import urlparse

from async_hdf5.store import HTTPStore

https_links = granule.data_links()
url = next(l for l in https_links if l.endswith(".h5"))
print(f"URL: {url}")

token = auth.token["access_token"]

parsed = urlparse(url)
dir_path, _, filename = parsed.path.rpartition("/")
base_url = f"{parsed.scheme}://{parsed.netloc}{dir_path}"

store = HTTPStore.from_url(
    base_url,
    client_options={"default_headers": {"Authorization": f"Bearer {token}"}},
)

## Open with xarray

async-hdf5 registers an xarray backend engine. We can open a specific HDF5
group — here, the `gt1l/land_ice_segments` beam — directly as an xarray
Dataset.

In [ ]:
import xarray as xr

ds = xr.open_dataset(
    filename,
    engine="async_hdf5",
    group="gt1l/land_ice_segments",
    store=store,
)
ds

## Read elevation data

The data is fetched lazily. Accessing `.values` triggers the actual
byte-range reads — decompression happens in zarr-python's codec pipeline.

In [ ]:
import numpy as np

h_li = ds["h_li"].values
lat = ds["latitude"].values
lon = ds["longitude"].values

# ATL06 uses 3.4028235e+38 as fill value; filter it out along with non-finite values
valid = np.isfinite(h_li) & (h_li < 1e10)
print(f"Shape: {h_li.shape}")
print(f"Valid: {valid.sum()}/{len(h_li)}")
print(f"Elevation range: {h_li[valid].min():.1f} to {h_li[valid].max():.1f} m")
print(f"Latitude range: {lat.min():.4f} to {lat.max():.4f}")

## Plot along-track elevation

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(lat[valid], h_li[valid], s=0.5, alpha=0.5)
ax.set_xlabel("Latitude")
ax.set_ylabel("Elevation (m)")
ax.set_title("ICESat-2 ATL06 — gt1l land ice elevation")
fig.tight_layout()

## Explore the full hierarchy with DataTree

ATL06 files have a deeply nested HDF5 group structure: 6 beam groups
(`gt1l` through `gt3r`), each with subgroups for segments, fit statistics,
geophysical corrections, and more. `open_datatree` exposes the full tree.

In [ ]:
dt = xr.open_datatree(filename, engine="async_hdf5", store=store)

for node in dt.subtree:
    n_vars = len(node.dataset.data_vars)
    if n_vars > 0:
        print(f"{node.path}: {n_vars} variables")

## Under the hood: low-level HDF5 inspection

The xarray engine is built on top of a lower-level async API that gives
direct access to HDF5 groups, datasets, attributes, and chunk indices.

In [ ]:
from async_hdf5 import HDF5File


async def inspect(store, path):
    f = await HDF5File.open(path, store=store)
    root = await f.root_group()

    # Navigate to a dataset
    gt1l = await root.group("gt1l")
    lis = await gt1l.group("land_ice_segments")
    h_li_ds = await lis.dataset("h_li")

    print("Dataset: h_li")
    print(f"  Shape: {h_li_ds.shape}")
    print(f"  Dtype: {h_li_ds.numpy_dtype}")
    print(f"  Chunk shape: {h_li_ds.chunk_shape}")
    print(f"  Filters: {h_li_ds.filters}")

    # Show chunk index — byte offsets into the remote file
    chunk_index = await h_li_ds.chunk_index()
    print(f"  Chunks in index: {len(list(chunk_index))}")


await inspect(store, filename)

Each chunk maps to a byte range in the remote HDF5 file. async-hdf5 parses
the HDF5 Extensible Array chunk index in Rust, then fetches only the
requested chunks — batching concurrent requests automatically.